# Démo : apprendre Black–Scholes avec des réseaux de neurones

Génération des données synthétiques, entraînement court, comparaison à une baseline, puis
quantification d'incertitude (ensemble et Monte-Carlo dropout).
Entrées : `m = S0/K`, `tau`, `r`, `sigma` ; cible : `C/K` (vérité terrain analytique).
Référence : van Mieghem, Papapantoleon & Papazoglou-Hennig (2023), arXiv:2307.07657.

Les sorties sont vidées avant commit ; exécuter *Run All* pour les régénérer (CPU).
Aucun chiffre n'est stocké dans le dépôt : les résultats (RMSE, couverture, std) se régénèrent uniquement en exécutant ce notebook.

In [ ]:
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from option_pricing.data import make_dataset
from option_pricing.evaluate import evaluate_model, metrics, predict
from option_pricing.mc_dropout import evaluate_mc, mc_dropout_predict
from option_pricing.train import train
from option_pricing.uncertainty import compare, train_ensemble

SEED = 42
# Overridable through the environment (the smoke test uses tiny values).
EPOCHS = int(os.environ.get("DEMO_EPOCHS", 30))
N_TRAIN = int(os.environ.get("DEMO_N_TRAIN", 5000))
N_EVAL = int(os.environ.get("DEMO_N_EVAL", 1000))
N_MODELS = int(os.environ.get("DEMO_N_MODELS", 5))
N_PASSES = int(os.environ.get("DEMO_N_PASSES", 100))

## 1. Données synthétiques
Split train/val/test tiré **une seule fois** ; le scaler est fit sur le train (dans `train`).

In [ ]:
ds = make_dataset(n_train=N_TRAIN, n_val=N_EVAL, n_test=N_EVAL, seed=SEED)
print({name: getattr(ds, name).X.shape for name in ("train", "val", "test")})
print(f"prix normalisé C/K : min {ds.train.y.min():.4f}, max {ds.train.y.max():.4f}")

## 2. Entraînement court (MLP, Adam)
Le meilleur epoch est choisi sur la validation uniquement.

In [ ]:
res = train(ds, "mlp", epochs=EPOCHS, lr=1e-3, optimizer="adam", seed=SEED)
plt.plot(res.train_loss, label="train")
plt.plot(res.val_loss, label="val")
plt.axvline(res.best_epoch, ls=":", c="gray")
plt.yscale("log")
plt.xlabel("epoch")
plt.ylabel("MSE")
plt.legend()
plt.show()

## 3. Comparaison à une baseline
Baseline : prédire la moyenne du prix d'entraînement. Les métriques sont calculées contre le vrai
prix BS. Le test set n'est utilisé qu'ici, pour l'évaluation finale.

In [ ]:
baseline = np.full_like(ds.test.y, ds.train.y.mean())
mlp_metrics = evaluate_model(res.model, ds.test)
rows = {
    "baseline (moyenne train)": metrics(ds.test.y, baseline),
    "MLP": mlp_metrics["all"],
}
pd.DataFrame(rows).T

In [ ]:
pred = predict(res.model, ds.test.X)
fig, ax = plt.subplots(1, 2, figsize=(10, 4))
ax[0].scatter(ds.test.y, pred, s=3, alpha=0.4)
ax[0].plot([0, 1], [0, 1], "r--")
ax[0].set_xlabel("prix BS")
ax[0].set_ylabel("prix MLP")
ax[1].hist(pred - ds.test.y, bins=50)
ax[1].set_xlabel("erreur (MLP - BS)")
plt.tight_layout()
plt.show()
pd.DataFrame(mlp_metrics).T

## 4. Incertitude : ensemble profond
5 réseaux (seeds différentes) ; l'écart-type entre membres est l'incertitude.
Les seeds sont 42-46 : le membre 0 est identique au MLP de la section 2 (réentraînement volontaire, pour garder la cellule autonome).

In [ ]:
ens = train_ensemble(ds, n_models=N_MODELS, epochs=EPOCHS, lr=1e-3, optimizer="adam", seed=SEED)
pd.DataFrame(compare(ens, ds.test)).T

## 5. Incertitude : Monte-Carlo dropout (T = 100)
Dropout actif à l'inférence. L'intervalle `mean ± 1.96·std` n'est **pas** calibré (cible
déterministe) : on le compare à un intervalle de largeur constante calé sur la validation.

In [ ]:
mc = evaluate_mc(res.model, ds.test, n_passes=N_PASSES, seed=SEED, calib=ds.val)
pd.Series(mc, name="MC dropout (test)").to_frame()

In [ ]:
mean, std = mc_dropout_predict(res.model, ds.test.X, n_passes=N_PASSES, seed=SEED)
err = np.abs(mean - ds.test.y)
plt.scatter(std, err, s=3, alpha=0.4)
plt.xlabel("std MC dropout")
plt.ylabel("|erreur|")
plt.show()